# Benchmarking propagation methods

In this notebook, we benchmark the various propagation methods `Euler`, `Expm`, `ExpmChebyshev`, `Vern7` and `DiffraxODE` for accuracy and runtime. We briefly discuss about choice of propagation methods for various scenarios. 

*Note: In this example we have explicitly not included `RungeKutta` as it does not support JIT compilation result in long runtime. Further, `DiffraxODE` and `Vern7` provide alternative ODE solver methods.*

In [ ]:
import time

import jax
import jax.numpy as jnp
import numpy as np
import plotting  # noqa: F401


def benchmark(function, repeats=3):
    """Time a function, separating the first (compiling) call from the fastest of the later ones.

    Returns the result of the function, the duration of the first call, and the fastest
    of ``repeats`` subsequent calls, both in seconds.
    """
    start = time.perf_counter()
    result = jax.block_until_ready(function())
    first_call = time.perf_counter() - start

    best = np.inf
    for _ in range(repeats):
        start = time.perf_counter()
        jax.block_until_ready(function())
        best = min(best, time.perf_counter() - start)

    return result, first_call, best

## Model

We consider a transmon <cite data-cite="koch2007charge">(Koch et al., 2007)</cite>, in a frame rotating with the drive, and driven by a Gaussian pulse on a slightly detuned local oscillator,

$$H(t) = \delta \, \hat{n} + \frac{\alpha}{2} \hat{n} (\hat{n} - 1) + \Omega(t) \, a + \Omega^*(t) \, a^\dagger .$$

In [ ]:
from paraqeet.eom.schroedinger_equation import SchroedingerEquation
from paraqeet.hamiltonian.drive import Drive
from paraqeet.hamiltonian.transmon import TransmonHamiltonian
from paraqeet.hamiltonian.utils import construct_basis_state
from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import GaussEnvelope
from paraqeet.signal.iq_mixer import IQMixer

t_final = 20e-9
detuning = 2 * np.pi * 50e6
anharm = -2 * np.pi * 200e6
num_levels = 3


def build_model(num_levels=3):
    """Return the envelope, the generator, the Hamiltonian and the equation of motion of a driven transmon."""
    tone = GaussEnvelope(amplitude=Quantity(np.pi / t_final, -4 * np.pi / t_final, 4 * np.pi / t_final))
    tone.t_final.set_value(t_final)
    generator = IQMixer(envelopes=[tone], frequency=Quantity(detuning, 0.0, 2 * detuning, unit="Hz"))

    hamiltonian = TransmonHamiltonian(
        num_levels=num_levels,
        frequency=Quantity(detuning, 0.0, 2 * detuning, unit="Hz"),
        anharmonicity=Quantity(anharm, 2 * anharm, 0.0, unit="Hz"),
        drives=[],
    )
    hamiltonian.drives = [Drive(hamiltonian.annihilation_op, generator, add_hermitian=True)]

    equation = SchroedingerEquation(
        hamiltonian_func=hamiltonian.get_value,
        hamiltonian_gradient_func=hamiltonian.get_gradient,
    )
    return tone, generator, hamiltonian, equation


tone, generator, hamiltonian, equation = build_model(num_levels)

times = np.array([0.0, t_final])
initial_state = construct_basis_state(dim=num_levels, index=0)
target_state = construct_basis_state(dim=num_levels, index=0)

## Propagation methods

Given the model, we define the propagation methods used for solving the dynamics. Here, in addition to fixed step size, we also add adaptive step size ODE solvers from Diffrax.

In [ ]:
import diffrax

from paraqeet.propagation import DiffraxODE, Euler, Expm, ExpmChebyshev, Vern7
from paraqeet.propagation.utils import schrodinger_step

adaptive_controller = diffrax.PIDController(rtol=1e-10, atol=1e-12)

# Needed for Tsit5 Adaptive
adaptive_max_steps = 200_000


def make_propagation(name, equation, resolution, initial_state):
    """Return a fresh propagation object of the given name."""
    if name == "Euler":
        return Euler(equation.get_value, resolution, initial_state)
    if name == "Expm":
        return Expm(equation.get_value, resolution, initial_state)
    if name == "ExpmChebyshev":
        return ExpmChebyshev(equation.get_value, resolution, initial_state, order=16)
    if name == "Vern7":
        return Vern7(equation.get_value, resolution, initial_state, step_function=schrodinger_step)
    if name == "Diffrax Tsit5":
        return DiffraxODE(equation.get_value, resolution, initial_state, step_function=schrodinger_step)
    if name == "Diffrax Tsit5 adaptive":
        return DiffraxODE(
            equation.get_value,
            resolution,
            initial_state,
            step_function=schrodinger_step,
            stepsize_controller=adaptive_controller,
            max_steps=adaptive_max_steps,
        )
    if name == "Diffrax Dopri8":
        return DiffraxODE(
            equation.get_value,
            resolution,
            initial_state,
            step_function=schrodinger_step,
            solver=diffrax.Dopri8(),
            interpolation_order=10,
        )
    if name == "Diffrax Dopri8 adaptive":
        return DiffraxODE(
            equation.get_value,
            resolution,
            initial_state,
            step_function=schrodinger_step,
            solver=diffrax.Dopri8(),
            interpolation_order=10,
            stepsize_controller=adaptive_controller,
            max_steps=adaptive_max_steps,
        )
    raise ValueError(f"Unknown propagation method {name}.")

We define `Vern7` at a very high resolution as our reference method.
As a verification, the resolution has to be high enough such that halving it leaves the state meaningfully unchanged.

In [ ]:
reference = np.array(make_propagation("Vern7", equation, 100e9, initial_state).get_value(times))[-1]
half = np.array(make_propagation("Vern7", equation, 50e9, initial_state).get_value(times))[-1]

print("Change when halving the resolution of the reference:", f"{np.linalg.norm(half - reference):.2e}")

Let's run all the propagation methods at various resolutions to benchmark the runtimes. Since all the propagation methods are JIT compiled, we store the fastest runtime (to avoid adding compile time)

In [ ]:
resolutions = [1e9, 2e9, 5e9, 10e9, 30e9, 50e9]
method_names = [
    "Euler",
    "Expm",
    "ExpmChebyshev",
    "Vern7",
    "Diffrax Tsit5",
    "Diffrax Tsit5 adaptive",
    "Diffrax Dopri8",
    "Diffrax Dopri8 adaptive",
]

errors = {name: [] for name in method_names}
runtimes = {name: [] for name in method_names}

print(f"{'method':>23s} | {'resolution':>12s} | {'error':>10s} | {'first call':>12s}  | {'fastest call':>14s}")
print("----------------------------------------------------------------------------------------------------")

for name in method_names:
    for resolution in resolutions:
        propagation = make_propagation(name, equation, resolution, initial_state)
        states, first_call, fastest = benchmark(lambda: propagation.get_value(times), repeats=2)
        error = float(np.linalg.norm(np.array(states)[-1] - reference))

        errors[name].append(error)
        runtimes[name].append(fastest)
        print(
            f"{name:>23s} | {resolution:12.1e} | {error:10.2e} | {first_call * 1e3:10.0f} ms | {fastest * 1e3:12.1f} ms"
        )
    print("----------------------------------------------------------------------------------------------------")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
markers = ["o", "s", "^", "D", "v", "P", "X", "*"]

for i, name in enumerate(method_names):
    axes[0].loglog(np.array(resolutions) / 1e9, errors[name], marker=markers[i], label=name)
    axes[1].loglog(np.array(resolutions) / 1e9, np.array(runtimes[name]) * 1e3, marker=markers[i], label=name)

axes[0].set_xlabel("Resolution [samples / ns]")
axes[0].set_ylabel("Error")
axes[1].set_xlabel("Resolution [samples / ns]")
axes[1].set_ylabel("Runtime [ms]")
for ax in axes:
    ax.grid(True, linestyle=(1, (1, 5)), linewidth=1)
axes[1].legend(loc="upper left", fontsize=8, frameon=True)
plt.tight_layout()
plt.show()

It can be seen that the `Euler` scales as $\mathcal{O}(\Delta t)$. `Expm` and `ExpmChebyshev` both saturate at order $\mathcal{O}(\Delta t^2)$, due to the midpoint interpolation. The ODE solvers have a much better scaling, with the adaptive solvers having an almost constant runtime across the resolutions.

If we create a runtime vs error plot, we can pick out the propagation methods that are faster with high accuracy

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, figsize=(6, 4))

for i, name in enumerate(method_names):
    axes.loglog(np.array(runtimes[name]) * 1e3, errors[name], marker=markers[i], ls="-", label=name)

axes.set_xlabel("Runtime [ms]")
axes.set_ylabel("Error")
axes.grid(True, linestyle=(1, (1, 5)), linewidth=1)
axes.legend(loc="upper right", fontsize=8, frameon=True)
plt.tight_layout()
plt.show()

It can be seen that there is a trade off between runtime and error (expect for the adaptive solvers): methods like `Vern7` and `Dopri8` provide the highest accuracy at the expense of wall-time and `Expm` based solvers are faster.

We can look at how the time is distributed per 100 steps of proapagation

In [ ]:
from paraqeet.propagation.utils import construct_times

resolution = 5e9
step_times, dt = construct_times(times, 1, resolution)
sample_times = step_times + dt / 2
steps = jnp.arange(len(step_times))
generators = jnp.array(equation.get_value(sample_times), dtype=jnp.complex128) * dt

_, _, t_signal = benchmark(lambda: generator.get_value(sample_times))
_, _, t_equation = benchmark(lambda: equation.get_value(sample_times))
_, _, t_propagate = benchmark(
    lambda: Expm._propagate(generators, jnp.array(initial_state, dtype=jnp.complex128), steps)
)

print(f"{len(step_times)} propagation steps at a resolution of {resolution:.0e} samples per second")
print(f"  signal generation      {t_signal * 1e3:8.2f} ms")
print(f"  equation of motion     {t_equation * 1e3:8.2f} ms")
print(f"  Expm propagation       {t_propagate * 1e3:8.2f} ms")

## Scaling with Hilbert space dimension

In [ ]:
dimensions = [3, 5, 10, 20, 30]
method_names = [
    "Expm",
    "ExpmChebyshev",
    "Vern7",
    "Diffrax Tsit5",
    "Diffrax Tsit5 adaptive",
    "Diffrax Dopri8",
    "Diffrax Dopri8 adaptive",
]

resolution = 50e9

runtimes = {name: [] for name in method_names}

print(f"{'method':>23s} | {'dimension':>10s} |  {'first call':>12s}  | {'fastest call':>14s}")
print("---------------------------------------------------------------------")

for name in method_names:
    for dimension in dimensions:
        _, _, _, equation = build_model(num_levels=dimension)
        initial_state = jnp.array(construct_basis_state(dimension, 0), dtype=jnp.complex128)
        propagation = make_propagation(name, equation, resolution, initial_state)
        states, first_call, fastest = benchmark(lambda: propagation.get_value(times), repeats=2)

        runtimes[name].append(fastest)
        print(f"{name:>23s} | {dimension:10d} | {first_call * 1e3:10.0f} ms | {fastest * 1e3:12.1f} ms")
    print("---------------------------------------------------------------------")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, figsize=(6, 4))

for i, name in enumerate(method_names):
    axes.loglog(dimensions, np.array(runtimes[name]) * 1e3, marker=markers[i], ls="-", label=name)

axes.set_xlabel("Dimension")
axes.set_ylabel("Runtime [ms]")
axes.grid(True, linestyle=(1, (1, 5)), linewidth=1)
axes.legend(loc="upper left", fontsize=8, frameon=True)
plt.tight_layout()
plt.show()

The advantage of adaptive stepping does not survive the growth of the Hilbert space, with the `max_steps` increased for `Tsit5`. And `ExpmChebyshev` provides the best scaling.